# 16 Normal Replacement RIB Prescreen

Controlled step only: resolve RIB visibility for replacement Normal candidates and produce UPDATE-ready candidates. This notebook does not download UPDATE files, create labels, extract features, or touch frozen roster/partition files.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from bgp_incident_detection.normal_rib_prescreen import run

PILOT = ROOT / 'data' / 'pilot'
candidates = pd.read_csv(PILOT / 'normal_replacement_candidates_v1.csv')
candidates

## Candidate Disposition

Candidates shorter than the existing 30-minute sustained-stability rule remain duration-ineligible. The rule is not changed here.

In [ ]:
duration_minutes = (pd.to_datetime(candidates['candidate_end'], utc=True) - pd.to_datetime(candidates['candidate_start'], utc=True)).dt.total_seconds() / 60
pd.DataFrame({
    'pre_screen_status': candidates['pre_screen_status'],
    'duration_bucket': pd.Series(['<30m' if value < 30 else '>=30m' for value in duration_minutes])
}).value_counts().reset_index(name='count')

## Run Prescreen

This command may download surrounding RIB snapshots under `data/raw_mrt/normal_pilot_rib/`, which is gitignored. It performs HEAD checks for planned UPDATE URLs but does not download UPDATE bodies.

In [ ]:
summary = run()
summary

## Outputs

In [ ]:
duration_disposition = pd.read_csv(PILOT / 'normal_replacement_duration_disposition_v1.csv')
rib_visibility = pd.read_csv(PILOT / 'normal_replacement_rib_visibility_v1.csv')
collector_selection = pd.read_csv(PILOT / 'normal_replacement_collector_selection_v1.csv')
update_ready = pd.read_csv(PILOT / 'normal_update_ready_candidates_v1.csv')
update_plan = pd.read_csv(PILOT / 'normal_update_ready_acquisition_plan_v1.csv')

display(duration_disposition)
display(rib_visibility[['candidate_period_id', 'collector', 'asn_visible_before', 'asn_visible_after', 'route_presence_stable', 'rib_visibility_status']])
display(collector_selection)
display(update_ready)
display(update_plan.groupby(['associated_incident_uid', 'associated_category', 'collector']).size().reset_index(name='planned_update_files'))

## Audit Checks

In [ ]:
assert len(duration_disposition) == 12
assert set(duration_disposition['duration_status']) == {'INELIGIBLE_DURATION'}
assert len(rib_visibility) == 4
assert set(rib_visibility['rib_visibility_status']) == {'PASS'}
assert len(update_ready) == 4
assert set(update_ready['final_prescreen_status']) == {'UPDATE_READY'}
assert len(update_plan) == 44
assert set(update_plan['remote_available'].astype(str)) == {'True'}
assert update_plan['notes'].str.contains('dry-run only, no UPDATE download').all()
print('PASS: replacement Normal RIB prescreen produced UPDATE-ready dry-run candidates without downloading UPDATE files.')